In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

import copy
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#Data Loading Class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles",
                                                dataName="EntrainmentTrackback_V2", #*testing
                                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#LOADING FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#####

#Import StatisticalFunctions 
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=dir2+'Functions/'
sys.path.append(path)

import StatisticalFunctions
from StatisticalFunctions import * # import NumericalFunctions 

In [ ]:
##############################################
#JOB ARRAY

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    num_jobs=ModelData.Ntime-1 #5min: 132; 3min: 220; 1min: 660
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
#Variable Data Functions

def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z','Y','X']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z,Y,X] = (dataDictionary[k] for k in variableNames)
    return Z,Y,X
def GetZ(t):    
    variableNames = ['Z']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z] = (dataDictionary[k] for k in variableNames)
    return [Z]
    
def GetLangrangianBinaryArray(t):
    Processed_String='PROCESSED_'
    variableNames=[f'{Processed_String}A_g',f'{Processed_String}A_c']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [A_g,A_c] = (dataDictionary[k] for k in variableNames)
    return A_g,A_c

In [ ]:
#Tracked Data Functions
def GetTrackedArrays():
    [trackedArrays,_]\
    = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,              
                                                          Results_InputOutput_Class,verbose=False)

    trackedArrays.pop('turbulentCL'); trackedArrays.pop('Thermal')

    parcelDepth=GetParcelDepth()
    for key1,inner1 in trackedArrays.items():
        if "SHALLOW" not in parcelDepth:
            inner1.pop('SHALLOW')
        if "DEEP" not in parcelDepth:
            inner1.pop('DEEP')
    return trackedArrays

def GetParcelDepth():
    return 'ALL'

In [ ]:
########################################
#MAIN CALCULATION FUNCTIONS

In [ ]:
#Entrainment Mask Calculation Functions
def CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, A1,A2, A1_Prior,A2_Prior,
                         isRemoveCondOrEvapEvents=True): 
    """
    Function to compute Lagrangian entrainment mask
    """
    
    #Calculation for Entrainment and Detrainment
    DMatrix_Entrainment = SubtractA(A2,A2_Prior)

    # Update D for entrainment/detrainment
    DMatrix_Entrainment[DMatrix_Entrainment < 0] = 0

    # Remove in-situ condensation/evaporation events
    if isRemoveCondOrEvapEvents:
        DMatrix_Entrainment=RemoveCondOrEvapEvents(DMatrix_Entrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)

    if t==0:
        entrainmentMask = np.zeros_like(A2, dtype=bool)
        return [entrainmentMask]
    else:
        entrainmentMask = DMatrix_Entrainment.astype(bool)
        return [entrainmentMask]

def SubtractA(A,A_Prior):
    D = np.zeros_like(A,dtype=np.int8)
    D = A*1 - A_Prior*1
    return D

def RemoveCondOrEvapEvents(array,
                           Z,Y,X,Z_Prior,Y_Prior,X_Prior):
    ParcelMoved = (Z != Z_Prior) | (Y != Y_Prior) | (X != X_Prior)
    array[~ParcelMoved]=0
    return array

def CalculateBothEntrainments(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior,
                              A_g,A_c,A_g_Prior,A_c_Prior):
    [entrainmentMask_g] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior,
                                             A1=A_c,A2=A_g,A1_Prior=A_c_Prior,A2_Prior=A_g_Prior)
    
    [entrainmentMask_c] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, 
                                             A1=A_g,A2=A_c,A1_Prior=A_g_Prior,A2_Prior=A_c_Prior)
    return [entrainmentMask_g,entrainmentMask_c]

In [ ]:
#Histogram Settings Functions

#Initialize Histogram Functions
def InitializeHistogramsDictionary(trackedArrays, varNames, trackTimes, property_bins, property_bins_Perturbation):
    """
    Initialize All Histograms in a Dictionary
    """
    histogramsDictionary = {}
    numTLevels = len(trackTimes)
    numZLevels = len(ModelData.zh[ModelData.zh < 6])
    parcelDepth = GetParcelDepth()

    for category in ['Any']+list(trackedArrays.keys()):
        histogramsDictionary[category] = {}
        histogramsDictionary[category][parcelDepth] = {}
        for updraftType in ["g","c"]:
            histogramsDictionary[category][parcelDepth][updraftType] = {}
            for varName in varNames + [a+'_prime' for a in varNames]:
                if varName.endswith('_prime'):
                    numVarBins = len(property_bins_Perturbation[varName]) - 1
                else:
                    numVarBins = len(property_bins[varName]) - 1
                histogramsDictionary[category][parcelDepth][updraftType][varName]\
                = MakeHistogram3D(numTLevels, numZLevels, numVarBins)
    return histogramsDictionary
    
def InitializeJointHistogramsDictionary(trackedArrays, varNames, trackTimes, z_bins, property_bins_Perturbation):
    """
    Initialize All Joint (QV_prime vs. other _prime variable) Histograms in a Dictionary.
    """
    histogramsDictionary = {}
    numTLevels = len(trackTimes)
    numZLevels = len(z_bins) - 1
    parcelDepth = GetParcelDepth()
    varPairs = GetJointVarPairs(varNames)
    for category in ['Any']+list(trackedArrays.keys()):
        histogramsDictionary[category] = {}
        histogramsDictionary[category][parcelDepth] = {}
        for updraftType in ["g","c"]:
            histogramsDictionary[category][parcelDepth][updraftType] = {}
            for varName_1, varName_2 in varPairs:
                pairKey = f"{varName_1}--{varName_2}"
                numVarBins_1 = len(property_bins_Perturbation[varName_1]) - 1
                numVarBins_2 = len(property_bins_Perturbation[varName_2]) - 1
                histogramsDictionary[category][parcelDepth][updraftType][pairKey]\
                = MakeJointHistogram4D(numTLevels, numZLevels, numVarBins_1, numVarBins_2)
    return histogramsDictionary

def GetDictionarySize(dictionary, format='GB'):
    """
    Recursively sums memory usage of a nested dict whose leaves are numpy arrays.
    """
    def GetBytes(d):
        totalBytes = 0
        if isinstance(d, dict):
            for value in d.values():
                totalBytes += GetBytes(value)
        elif isinstance(d, np.ndarray):
            totalBytes += d.nbytes
        return totalBytes

    totalBytes = GetBytes(dictionary)

    if format.upper() in ['G','GB']:
        return totalBytes / 1e9
    elif format.upper() in ['M','MB']:
        return totalBytes / 1e6
def CheckOutputSize(histogramsDictionary,jointHistogramsDictionary):
    totalBytes = GetDictionarySize(histogramsDictionary)
    print(f'histogramsDictionary: {totalBytes:.3f} GB')
    totalBytes_joint = GetDictionarySize(jointHistogramsDictionary)
    print(f'jointHistogramsDictionary: {totalBytes_joint:.3f} GB')
    
def MakeHistogram3D(numTLevels, numZLevels, numVarBins):
    """
    Create an empty 3D histogram array with axes (time, z, variable property).
    """
    histogram3D = np.zeros((numTLevels, numZLevels, numVarBins), dtype=np.int32)
    return histogram3D
def MakeJointHistogram4D(numTLevels, numZLevels, numVarBins_1, numVarBins_2):
    """
    Create an empty 4D joint histogram array with axes (time, z, property_1, property_2),
    using int32 since these are exact counts, not continuous values.
    """
    jointHistogram4D\
    = np.zeros((numTLevels, numZLevels, numVarBins_1, numVarBins_2), dtype=np.int32)
    return jointHistogram4D

#Histogram Binning Functions

def BinHistogram(histogramsDictionary, varName, relative_time,
            entrainedVariableData, entrainedVariableData_t, entrainedZ,
            parcelType, parcelDepth, updraftType,
            time_bins, z_bins,
            property_bins, property_bins_Perturbation):
    """
    Bins already-masked entrained parcel data (raw and perturbation) into a 3D (time, z, property) histogram.
    """
    #Non-Perturbation Values
    countsRaw, _ = np.histogramdd(
        sample=np.column_stack([
            np.full_like(entrainedVariableData, relative_time),
            entrainedZ,
            entrainedVariableData]),
        bins=[time_bins, z_bins, property_bins[varName]] #(time, z, property)
    )
    histogramsDictionary[parcelType][parcelDepth][updraftType][varName]\
    += countsRaw.astype(np.int32)

    #Perturbation Values (value at t_back minus value at t, for the same parcels)
    primeVarName = varName+'_prime'
    entrainedVariableData_prime = entrainedVariableData - entrainedVariableData_t
    countsPrime, _ = np.histogramdd(
        sample=np.column_stack([
            np.full_like(entrainedVariableData_prime, relative_time),
            entrainedZ,
            entrainedVariableData_prime
        ]),
        bins=[time_bins, z_bins, property_bins_Perturbation[primeVarName]] #(time, z, property)
    )
    histogramsDictionary[parcelType][parcelDepth][updraftType][primeVarName]\
    += countsPrime.astype(np.int32)
    return histogramsDictionary

def BinJointHistogram(histogramsDictionary, varName_1, varName_2, relative_time,
                       entrainedVariableData_1, entrainedVariableData_1_t,
                       entrainedVariableData_2, entrainedVariableData_2_t,
                       entrainedZ,
                       parcelType, parcelDepth, updraftType,
                       time_bins, z_bins,
                       property_bins_Perturbation):
    primeVarName_1 = varName_1+'_prime'
    primeVarName_2 = varName_2+'_prime'

    entrainedVariableData_1_prime = entrainedVariableData_1 - entrainedVariableData_1_t
    entrainedVariableData_2_prime = entrainedVariableData_2 - entrainedVariableData_2_t

    pairKey = f"{primeVarName_1}--{primeVarName_2}"
    counts, _ = np.histogramdd(
        sample=np.column_stack([
            np.full_like(entrainedVariableData_1_prime, relative_time),
            entrainedZ,
            entrainedVariableData_1_prime,
            entrainedVariableData_2_prime]),
        bins=[time_bins, z_bins, property_bins_Perturbation[primeVarName_1], property_bins_Perturbation[primeVarName_2]] #(time, z, property_1, property_2)
    )
    histogramsDictionary[parcelType][parcelDepth][updraftType][pairKey] += counts.astype(np.int32)
    return histogramsDictionary

In [ ]:
#Histogram Variable Bin Settings
def GetVarNames():
    varNames = ["W"]
    varNames += ["QV","RH_vapor","QCQI"]
    varNames += ["T",'THETA_v']
    return varNames
    
def GetJointVarPairs(varNames):
    """
    Pairs QV against every other variable, plus the same pairing for the _prime versions.
    """
    varPairs = [("QV_prime", varName+"_prime") for varName in varNames if varName != "QV"]
    # varPairs += [("QV", varName) for varName in varNames if varName != "QV"]
    return varPairs

def GetNumBins():
    numBins=300 
    return numBins
def GetNumBins_Joint():
    numBins=100
    return numBins
    
def GetPropertyBins(numBins=None):
    """
    Bins for histogram of raw variable values (no baseline subtraction).
    #CheckRanges: these are rough atmospheric defaults - confirm against your simulation's actual value ranges
    """
    
    if numBins is None:
        numBins = GetNumBins()
    
    property_bins_Dictionary = {
        "QV":       np.linspace(10/1e3, 20/1e3, numBins+1),   # kg/kg
        "RH_vapor": np.linspace(30/1e2, 100/1e2, numBins+1),        # %
        "QCQI":     np.linspace(0, 3/1e3, numBins+1),      # kg/kg
        "W":        np.linspace(-10, 10, numBins+1),       # m/s
        "T":        np.linspace(280, 315, numBins+1),      # K
        "THETA_v":  np.linspace(300, 320, numBins+1),      # K
    }
    return property_bins_Dictionary

def GetPropertyBins_Perturbations(numBins=None):
    """
    Bins for histogram of perturbations
    defined as value minus value at time of entrainment
    """

    if numBins is None:
        numBins = GetNumBins()
    
    property_bins_Dictionary = {
        "QV":    MakeSignedLogBins(None, 5e-5, 5/1e3, numBins),
        "RH_vapor": MakeSignedLogBins(None, 0.01/1e2, 2/1e2, numBins), 
        "QCQI":  MakeSignedLogBins(None, 1e-8, 2/1e3, numBins),
        "W":     MakeSignedLogBins(None, 0.02, 15, numBins),
        "T":     MakeSignedLogBins(-10, 0.01, 30, numBins),
        "THETA_v":    MakeSignedLogBins(None, 0.1, 10, numBins),
    }
    
    for varName in list(property_bins_Dictionary.keys()):
        property_bins_Dictionary[varName+"_prime"] = property_bins_Dictionary[varName]
    return property_bins_Dictionary
    
def MakeSignedLogBins(minValue=-1, centerValue=0.01, maxValue=1, nBins=500):
    """
    Create symmetric signed-logarithmic bins including zero.
    minValue: negative endpoint (e.g. -20) — most extreme negative bin edge.
    centerValue: small positive magnitude closest to zero on both sides (e.g. 0.01).
    maxValue: positive endpoint (e.g. 30) — most extreme positive bin edge.
    """
    if nBins % 2 != 0:
        raise ValueError("nBins must be even for signed-log bins")
    if minValue is not None and minValue >= 0:
        raise ValueError("minValue must be negative")
    halfBins = nBins // 2
    
    # positive side (log-spaced, excludes zero): centerValue -> maxValue
    pos = np.logspace(
        np.log10(centerValue),
        np.log10(maxValue),
        halfBins
    )
    # negative side mirror
    if minValue is None:
        neg = -pos[::-1]
    else:
        # negative side (log-spaced, excludes zero): centerValue -> abs(minValue)
        neg = -np.logspace(
            np.log10(centerValue),
            np.log10(-minValue),
            halfBins
        )[::-1]
        
    # form bins with 0 in the center
    bins = np.concatenate([neg, [0.0], pos])
    return bins

In [ ]:
#Preconditioning Binning Functions
def GetTrackedParcelIndices(entrainmentMask, Z, Y, X,
                             trackedArrays, t, parcelType='Any'):
    """
    Computed ONCE at reference time t. Returns the fixed set of parcel
    indices (as a boolean mask over Np) that are 'entrained at t' (and,
    for parcelType != 'Any', share a gridbox with a tracked parcel at t).
    This selection does NOT change as t_back varies in the backward loop.
    """
    if parcelType == 'Any':
        return entrainmentMask

    trackedParcelList = trackedArrays[parcelType][GetParcelDepth()]
    pIndex = trackedParcelList[:,0]
    t1 = trackedParcelList[:,1]
    t2 = np.minimum(trackedParcelList[:,2] + trackedParcelList[:,3], ModelData.Ntime) 

    withinTimeInterval = np.zeros(ModelData.Np, dtype=bool)
    withinTimeInterval[pIndex] = (t >= t1) & (t <= t2)   #*** t, not t_back ***
    trackedPs = np.where(withinTimeInterval)[0]

    cellID = np.ravel_multi_index((Z, Y, X), (ModelData.Nzh, ModelData.Nyh, ModelData.Nxh))
    trackedCellIDs = cellID[trackedPs]
    sharesGridbox = np.isin(cellID, trackedCellIDs)
    sharesGridbox[trackedPs] = False

    return entrainmentMask & sharesGridbox

def GetCoordinateBins():

    #Time Bins
    timesteps_per_min = 1/(ModelData.time[1].item()/1e9/60)
    timesteps_per_hour = int(60*timesteps_per_min)
    # time_bins = np.arange(0,(0-timesteps_per_hour)-1,-1)[::-1]
    time_bins = np.arange(0.5, -timesteps_per_hour-1.5, -1)[::-1]

    numLevelsBelow6km = np.sum(ModelData.zh < 6)
    z_bins = np.arange(-0.5, numLevelsBelow6km + 0.5, 1)

    return (time_bins,timesteps_per_hour,z_bins)

In [ ]:
#Calculation Functions
def RunCode():
    """
    Coordinates 60 minute trackback binning of entrained parcel data
    for for all parcelTypes (i.e. 'CL','nonCL','SBF','ColdPool')
    """

    (histogramsDictionary,jointHistogramsDictionary)=(None,None)
    
    #Getting VariableNames and Bins
    varNames=GetVarNames()
    property_bins=GetPropertyBins() 
    property_bins_Perturbation=GetPropertyBins_Perturbations()
    property_bins_Joint_Perturbation\
    =GetPropertyBins_Perturbations(numBins=GetNumBins_Joint())

    [time_bins,timesteps_per_hour,
     z_bins]=GetCoordinateBins()
    
    #Get Tracked Parcel Data
    trackedArrays=GetTrackedArrays()
    
    #Running Through Each Time
    for t in tqdm(loop_elements, desc="Processing"):
        if ModelData.time_hrs[t]<10: #LT
            print(f"skipping time {t} at {ModelData.time_hrs[t]} is before 10 LT")
            continue
        
        #getting variable data
        [Z,Y,X] = GetSpatialData(t)
        [Z_Prior,Y_Prior,X_Prior] = GetSpatialData(t-1)
        [A_g,A_c] = GetLangrangianBinaryArray(t)
        [A_g_Prior,A_c_Prior] = GetLangrangianBinaryArray(t-1)
        [entrainmentMask_g,entrainmentMask_c]\
        =CalculateBothEntrainments(t,
                                   Z,Y,X,Z_Prior,Y_Prior,X_Prior,
                                   A_g,A_c,A_g_Prior,A_c_Prior)

        fixedMasks = {}
        for parcelType in ['Any']+list(trackedArrays.keys()):
            mask_g = GetTrackedParcelIndices(entrainmentMask_g, Z,Y,X,
                                             trackedArrays, t, parcelType)
            mask_c = GetTrackedParcelIndices(entrainmentMask_c, Z,Y,X,
                                             trackedArrays, t, parcelType)
            fixedMasks[parcelType] = {"g": mask_g, "c": mask_c}
            
        trackTimes = np.arange(t,(t-timesteps_per_hour*1)-1,-1)
        histogramsDictionary\
        = InitializeHistogramsDictionary(trackedArrays, varNames, 
                                         trackTimes, 
                                         property_bins,
                                         property_bins_Perturbation)
        jointHistogramsDictionary\
        = InitializeJointHistogramsDictionary(trackedArrays, varNames,   
                                              trackTimes, z_bins,        
                                              property_bins_Joint_Perturbation) 

        TrackBackData(t,varNames,histogramsDictionary,jointHistogramsDictionary,
                      Z,Y,X,
                      trackedArrays,fixedMasks,
                      trackTimes,time_bins,z_bins,
                      property_bins,property_bins_Perturbation,
                      property_bins_Joint_Perturbation)

        #Save Data 
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
                                                      histogramsDictionary,
                                                      dataName="EntrainmentTrackback", t=t)
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
                                                      jointHistogramsDictionary,
                                                      dataName=("EntrainmentTrackback"
                                                                "_JointDistributions"), t=t)
        CheckOutputSize(histogramsDictionary,jointHistogramsDictionary)
    
    return (histogramsDictionary,jointHistogramsDictionary)
    
            

def TrackBackData(t,varNames,histogramsDictionary,jointHistogramsDictionary, 
                  Z,Y,X,
                  trackedArrays,fixedMasks,
                  trackTimes,time_bins,z_bins,
                  property_bins,property_bins_Perturbation,
                  property_bins_Joint_Perturbation):
    
    #backwards loop here
    varDictionary_t = MakeDataDictionary(varNames, t)
    for count, t_back in enumerate(tqdm(trackTimes,desc="\t\tTracking back parcels",leave=False)):
        relative_time = t_back - t
        
        varDictionary = MakeDataDictionary(varNames, t_back) #get all variables at specified time
        
        BinVariablesToHistogram(varDictionary,varDictionary_t,trackedArrays,
                                t_back,Z,Y,X,relative_time,
                                histogramsDictionary,
                                fixedMasks,
                                time_bins,z_bins,
                                property_bins,property_bins_Perturbation)
        
        BinVariablesToJointHistogram(varDictionary,varDictionary_t,trackedArrays,varNames,
                                     t_back,Z,Y,X,relative_time,
                                     jointHistogramsDictionary,
                                     fixedMasks,
                                     time_bins,z_bins,
                                     property_bins_Joint_Perturbation)

def BinVariablesToHistogram(varDictionary, varDictionary_t, trackedArrays,
                            t_back, Z, Y, X, relative_time,
                            histogramsDictionary,
                            fixedMasks,
                            time_bins, z_bins,
                            property_bins, property_bins_Perturbation):
    parcelDepth = GetParcelDepth()
    for varName in varDictionary.keys():
        variableData = varDictionary[varName]
        variableData_t = varDictionary_t[varName]
        for parcelType in ['Any']+list(trackedArrays.keys()):
            mask_g = fixedMasks[parcelType]["g"]
            mask_c = fixedMasks[parcelType]["c"]
            for updraftType, mask in [("g", mask_g), ("c", mask_c)]:
                BinHistogram(histogramsDictionary,varName,relative_time,
                        variableData[mask], variableData_t[mask], Z[mask],
                        parcelType, parcelDepth, updraftType,
                        time_bins, z_bins,
                        property_bins, property_bins_Perturbation) #bin the data into histogramsDictionary
    return histogramsDictionary

def BinVariablesToJointHistogram(varDictionary, varDictionary_t, trackedArrays, varNames,
                                 t_back, Z, Y, X, relative_time,
                                 jointHistogramsDictionary,
                                 fixedMasks,
                                 time_bins, z_bins,
                                 property_bins_Perturbation):
    parcelDepth = GetParcelDepth()
    for varName_1, varName_2 in GetJointVarPairs(varNames):
        baseVarName_1 = varName_1.replace('_prime',''); baseVarName_2 = varName_2.replace('_prime','')
        variableData_1 = varDictionary[baseVarName_1]; variableData_1_t = varDictionary_t[baseVarName_1]
        variableData_2 = varDictionary[baseVarName_2]; variableData_2_t = varDictionary_t[baseVarName_2]
        for parcelType in ['Any']+list(trackedArrays.keys()):
            mask_g = fixedMasks[parcelType]["g"]
            mask_c = fixedMasks[parcelType]["c"]
            for updraftType, mask in [("g", mask_g), ("c", mask_c)]:
                BinJointHistogram(jointHistogramsDictionary, baseVarName_1, baseVarName_2, relative_time,
                        variableData_1[mask], variableData_1_t[mask],
                        variableData_2[mask], variableData_2_t[mask],
                        Z[mask],
                        parcelType, parcelDepth, updraftType,
                        time_bins, z_bins,
                        property_bins_Perturbation)
    return jointHistogramsDictionary

In [ ]:
########################################
#RUNNING
running=True
running=False

In [ ]:
#Running
if running:
    # loop_elements=[100] #*testing
    (histogramsDictionary,jointHistogramsDictionary) = RunCode()

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombining=True
recombining=False

In [ ]:
def RecombineProfiles(ModelData, DataManager, dataName):
    """
    Combine tracked-profile histogram dictionaries across all timesteps,
    using the first available timestep as a template. Works for both the
    single-variable (histogramsDictionary) and joint (jointHistogramsDictionary)
    structures, since both share the same [category][parcelDepth][updraftType][key]
    nesting with numpy arrays as leaves.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files ({dataName})...\n")

    [_,timesteps_per_hour,_]=GetCoordinateBins()

    combinedDictionary = None
    for t in tqdm(range(ModelData.Ntime), desc="Combining Profiles", unit="timestep"):

        if t <= timesteps_per_hour:
            # print(f"skipping time {t} since too close to first hour")
            continue

        try:
            histogramsDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(
                ModelData, DataManager, dataName=dataName, t=t)
        except FileNotFoundError:
            continue

        if combinedDictionary is None:
            combinedDictionary = copy.deepcopy(histogramsDictionary)
            continue

        AccumulateIntoCombined(combinedDictionary, histogramsDictionary)

    return combinedDictionary

def AccumulateIntoCombined(combinedDictionary, histogramsDictionary):
    """
    Recursively adds histogramsDictionary's numpy-array leaves into combinedDictionary,
    in place, walking down through nested dicts of any depth.
    """
    for key in histogramsDictionary:
        value = histogramsDictionary[key]
        if isinstance(value, dict):
            AccumulateIntoCombined(combinedDictionary[key], value)
        else:
            combinedDictionary[key] += value
    return combinedDictionary

In [ ]:
#Recombining
if recombining and not running:
    histogramsDictionary_combined = RecombineProfiles(ModelData, DataManager_TrackedProfiles, dataName="EntrainmentTrackback")
    TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
                                                  histogramsDictionary_combined, dataName="EntrainmentTrackback", t='combined')

    jointHistogramsDictionary_combined = RecombineProfiles(ModelData, DataManager_TrackedProfiles, dataName="EntrainmentTrackback_JointDistributions")
    TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
                                                  jointHistogramsDictionary_combined, dataName="EntrainmentTrackback_JointDistributions", t='combined')

In [ ]:
###################
#PLOTTING

In [ ]:
###################
#LOADING BACK IN

In [ ]:
# Loading Back In
if not running:
    histogramsDictionary_combined = TrackedProfiles_DataLoading_CLASS.LoadProfile(
        ModelData, DataManager_TrackedProfiles, dataName="EntrainmentTrackback", t='combined')
    
    jointHistogramsDictionary_combined = TrackedProfiles_DataLoading_CLASS.LoadProfile(
        ModelData, DataManager_TrackedProfiles, dataName="EntrainmentTrackback_JointDistributions", t='combined')

In [ ]:
###################
#PLOTTING FUNCTIONS

In [ ]:
# Plotting Font and Variable Settings

# Plotting Font Settings
plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})

# Plotting Variable Settings
def GetVarNames_Plotting():
    varNames=GetVarNames()
    varsToRemove=['QCQI','RH_vapor']
    for varToRemove in varsToRemove:
        varNames.remove(varToRemove)
    return varNames

labelsDictionary = {"QV": r"$q_v$",
                    "RH_vapor": r"$RH$",
                    "QCQI": r"$q_c + q_i$",
                    "W": r"$w$",
                    "T": r"$T$",
                    "THETA_v": r"$\theta_{\rho}$",}

labelsDictionaryPerturbation = {"QV": r"$q_v'$",
                                "RH_vapor": r"$RH'$",
                                "QCQI": r"$q_c' + q_i'$",
                                "W": r"$w'$",
                                "T": r"$T'$" ,
                                "THETA_v": r"$\theta_{\rho}'$"}

unitsDictionary = {"QV": "g/kg", 
                   "RH_vapor": "%",
                   "QCQI": "g/kg", 
                   "W": "m/s", 
                   "T": "K",
                   "THETA_v": "K"}

multiplierDictionary = {"QV": 1e3, 
                        "RH_vapor": 1e2,
                        "QCQI": 1e3}



In [ ]:
#Slicing/Averaging Helper Functions

def GetTimeIndexRange(time_bins, t1_min, t2_min, timesteps_per_min):
    """
    Returns the array index range into the time axis corresponding to
    the minute range [t1_min, t2_min] back from t=0 (exclusive of 0 itself
    unless t1_min=0 is explicitly wanted).
    """
    idx1 = len(time_bins) - 1 - int(timesteps_per_min * t2_min) - 1
    idx2 = len(time_bins) - 1 - int(timesteps_per_min * t1_min)
    return idx1, idx2

def GetZIndexRange(z_bins, z1_km, z2_km):
    """
    Returns the array index range into the z axis corresponding to
    the height range [z1_km, z2_km], using ModelData.zh to convert
    grid-level index back to physical height.
    """
    z_centers = 0.5*(z_bins[:-1] + z_bins[1:])
    zHeights_km = ModelData.zh[z_centers.astype(int)]
    idx = np.where((zHeights_km >= z1_km) & (zHeights_km < z2_km))[0]
    if idx.size == 0:
        return None, None
    return idx[0], idx[-1]+1

def SumOverZRange(histogram3D, z_bins, z1_km, z2_km):
    """
    Collapses a (time, z, property) histogram into (time, property),
    summing over the z-index range corresponding to [z1_km, z2_km].
    """
    idx1, idx2 = GetZIndexRange(z_bins, z1_km, z2_km)
    if idx1 is None:
        return None
    return histogram3D[:, idx1:idx2, :].sum(axis=1)

def SumOverTimeRange(histogram3D, time_bins, t1_min, t2_min, timesteps_per_min):
    """
    Collapses a (time, z, property) histogram into (z, property),
    summing over the time-index range corresponding to [t1_min, t2_min] minutes back.
    """
    idx1, idx2 = GetTimeIndexRange(time_bins, t1_min, t2_min, timesteps_per_min)
    return histogram3D[idx1:idx2, :, :].sum(axis=0)

In [ ]:
def CombinedPlot_PropertyHistogram(histogramsDictionary_combined, updraftType, z1_km, z2_km,
                                    parcelTypes=["CL", "nonCL"],
                                    plotType="contour", normalize=True, num_levels=20,
                                    plotPerturbations=True,
                                    labelsDictionary=None):
    """
    Plots property histograms either as perturbations from the entrainment-time
    baseline (plotPerturbations=True) or as the variables' own values (plotPerturbations=False),
    summed over the z-range [z1_km, z2_km).
    """
    if labelsDictionary is None:
        labelsDictionary = labelsDictionaryPerturbation if plotPerturbations else globals()["labelsDictionary"]

    mins = ModelData.time[1].item() / 1e9 / 60
    [time_bins, timesteps_per_hour, z_bins] = GetCoordinateBins()
    timesteps_per_min = timesteps_per_hour/60
    if plotPerturbations:
        property_bins_Dictionary = GetPropertyBins_Perturbations()
        property_limits_Dictionary = GetPropertyLimits_Perturbations(property_bins_Dictionary, updraftType)
    else:
        property_bins_Dictionary = GetPropertyBins()
        property_limits_Dictionary = GetPropertyLimits(property_bins_Dictionary,updraftType)
    varNames     = GetVarNames_Plotting()
    parcelDepth = GetParcelDepth()
    colorbarLabel = "Frequency (%)" if normalize else "Count"
    cmap = plt.get_cmap("turbo").copy()
    cmap.set_under("white")

    nrows = len(parcelTypes)
    ncols = len(varNames)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.5 * ncols, 3.5 * nrows),
        constrained_layout=True
    )
    axes = np.atleast_2d(axes)

    plotObjects_by_col = {j: [] for j in range(ncols)}
    vmin_by_col = {j: np.inf  for j in range(ncols)}
    vmax_by_col = {j: -np.inf for j in range(ncols)}

    # First pass: compute global vmin/vmax per column
    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):
            histKey = varName+'_prime' if plotPerturbations else varName
            histogram3D = histogramsDictionary_combined[parcelType][parcelDepth][updraftType][histKey]
            a = SumOverZRange(histogram3D, z_bins, z1_km, z2_km)
            if a is None:
                continue
            if normalize:
                a = NormalizeHistogram_Perturbations(a)
                a *= 100
            col_min = np.nanmin(a[:-1, :])
            col_max = np.nanmax(a[:-1, :])
            if np.isfinite(col_min):
                vmin_by_col[j] = min(vmin_by_col[j], col_min)
            if np.isfinite(col_max):
                vmax_by_col[j] = max(vmax_by_col[j], col_max)

    # Second pass: plot with shared scale
    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):
            ax = axes[i, j]
            ax.set_rasterization_zorder(1)
            histKey = varName+'_prime' if plotPerturbations else varName
            histogram3D = histogramsDictionary_combined[parcelType][parcelDepth][updraftType][histKey]
            a = SumOverZRange(histogram3D, z_bins, z1_km, z2_km)

            vmin = vmin_by_col[j]
            vmax = vmax_by_col[j]

            if a is None or not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
                ax.set_visible(False)
                continue

            if normalize:
                a = NormalizeHistogram_Perturbations(a)
                a *= 100

            levels = np.linspace(vmin, vmax, num_levels)

            x = mins * time_bins
            y = property_bins_Dictionary[varName]
            x_centers = 0.5 * (x[:-1] + x[1:])
            y_centers = 0.5 * (y[:-1] + y[1:])
            X, Y = np.meshgrid(x_centers, y_centers)
            multiplier = multiplierDictionary.get(varName, 1)
            units = unitsDictionary.get(varName, "")

            if plotType == "contour":
                plotObject = ax.contourf(
                    X, multiplier * Y, a.T,
                    cmap=cmap, levels=levels,
                    vmin=vmin, vmax=vmax, extend='min'
                )
            else:
                plotObject = ax.pcolormesh(
                    x, multiplier * y, a.T,
                    cmap=cmap, shading="auto",
                    vmin=vmin, vmax=vmax
                )

            ymin, ymax = property_limits_Dictionary[varName]
            ax.set_ylim(ymin, ymax)

            plotObjects_by_col[j].append(plotObject)

            if j == 0:
                ax.set_ylabel(f"{parcelType}\n{labelsDictionary[varName]} ({units})")
            else:
                ax.set_ylabel(fr"{labelsDictionary[varName]} ({units})")
            # if i == 0:
            #     ax.set_title(labelsDictionary[varName])
            if i == nrows - 1:
                ax.set_xlabel("Backwards Time (mins)")

    for j in range(ncols):
        if not plotObjects_by_col[j]:
            continue
        plt.colorbar(
            plotObjects_by_col[j][0],
            ax=axes[:, j],
            location="right",
            shrink=0.8,
            label=colorbarLabel if j == ncols - 1 else ""
        )

    updraftType = "General" if updraftType == 'g' else "Cloudy"
    plt.suptitle(
        f"History of Entrained Parcels ({updraftType} Updrafts) ({z1_km}-{z2_km}km)",
        fontsize=16
    )

    if plotPerturbations:
        for axis in axes.flat:
            axis.axhline(0, color='black', linestyle='--', alpha=0.6, linewidth=1)
    return fig

def CombinedPlot_PropertyHistogram_ZSubsetting(
        histogramsDictionary_combined, updraftType, t1_min, t2_min,
        parcelTypes=["CL", "nonCL"],
        plotType="contour", normalize=True, num_levels=20,
        plotPerturbations=True,
        labelsDictionary=None):
    """
    Same as CombinedPlot_PropertyHistogram but summed over the time-range
    [t1_min, t2_min] minutes back, plotted against z on the y-axis.
    """
    if labelsDictionary is None:
        labelsDictionary = labelsDictionaryPerturbation if plotPerturbations else globals()["labelsDictionary"]
 
    [time_bins, timesteps_per_hour, z_bins] = GetCoordinateBins()
    timesteps_per_min = timesteps_per_hour/60
    if plotPerturbations:
        property_bins_Dictionary = GetPropertyBins_Perturbations()
        property_limits_Dictionary = GetPropertyLimits_Perturbations(property_bins_Dictionary, updraftType)
    else:
        property_bins_Dictionary = GetPropertyBins()
        property_limits_Dictionary = GetPropertyLimits(property_bins_Dictionary, updraftType)
    varNames     = GetVarNames_Plotting()
    parcelDepth = GetParcelDepth()
    colorbarLabel = "Frequency (%)" if normalize else "Count"
    cmap = plt.get_cmap("turbo").copy()
    cmap.set_under("white")
 
    nrows = len(parcelTypes)
    ncols = len(varNames)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.5 * ncols, 3.5 * nrows),
        constrained_layout=True
    )
    axes = np.atleast_2d(axes)
 
    plotObjects_by_col = {j: [] for j in range(ncols)}
    vmin_by_col = {j: np.inf  for j in range(ncols)}
    vmax_by_col = {j: -np.inf for j in range(ncols)}
 
    for parcelType in parcelTypes:
        for j, varName in enumerate(varNames):
            histKey = varName+'_prime' if plotPerturbations else varName
            histogram3D = histogramsDictionary_combined[parcelType][parcelDepth][updraftType][histKey]
            a = SumOverTimeRange(histogram3D, time_bins, t1_min, t2_min, timesteps_per_min)
            if normalize:
                a = NormalizeHistogram_Perturbations_ZSubsetting(a)
                a *= 100
            vmin_by_col[j] = min(vmin_by_col[j], np.nanmin(a[:-1, :]))
            vmax_by_col[j] = max(vmax_by_col[j], np.nanmax(a[:-1, :]))
 
    zLevelTop = 2 if updraftType == "g" else 6
    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):
            ax = axes[i, j]
            ax.set_rasterization_zorder(1)
            histKey = varName+'_prime' if plotPerturbations else varName
            histogram3D = histogramsDictionary_combined[parcelType][parcelDepth][updraftType][histKey]
            a = SumOverTimeRange(histogram3D, time_bins, t1_min, t2_min, timesteps_per_min)
            if normalize:
                a = NormalizeHistogram_Perturbations_ZSubsetting(a)
                a *= 100
 
            vmin = vmin_by_col[j]
            vmax = vmax_by_col[j]
            levels = np.linspace(vmin, vmax, num_levels)
            if np.all(levels == 0):
                levels = num_levels
 
            x = property_bins_Dictionary[varName]
            y = z_bins
            x_centers = 0.5 * (x[:-1] + x[1:])
            y_centers = 0.5 * (y[:-1] + y[1:])
            y_centers = ModelData.zh[np.clip(y_centers.astype(int), 0, len(ModelData.zh) - 1)]
            X, Y = np.meshgrid(x_centers, y_centers)
            multiplier = multiplierDictionary.get(varName, 1)
            units = unitsDictionary.get(varName, "")
 
            if plotType == "contour":
                plotObject = ax.contourf(
                    multiplier * X, Y, a,
                    cmap=cmap, levels=levels,
                    vmin=vmin, vmax=vmax, extend='min')
            else:
                plotObject = ax.pcolormesh(
                    multiplier * x, y, a,
                    cmap=cmap, shading="auto",
                    vmin=vmin, vmax=vmax, extend='min')
 
            xmin, xmax = property_limits_Dictionary[varName]
            ax.set_xlim(xmin, xmax)
 
            plotObjects_by_col[j].append(plotObject)
 
            if j == 0:
                ax.set_ylabel(f"{parcelType}\nz (km)")
            else:
                ax.set_ylabel("z (km)")
            if i == 0:
                ax.set_title(labelsDictionary[varName])
            if i == nrows - 1:
                ax.set_xlabel(fr"{labelsDictionary[varName]} ({units})")
            ax.set_ylim(0, zLevelTop)
 
    for j in range(ncols):
        plt.colorbar(
            plotObjects_by_col[j][0],
            ax=axes[:, j],
            location="right",
            shrink=0.8,
            label=colorbarLabel
        )
 
    updraftType = "General" if updraftType == 'g' else "Cloudy"
    plt.suptitle(
        f"History of Entrained Parcels ({updraftType} Updrafts) ({t1_min}-{t2_min}mins)",
        fontsize=16
    )
 
    for ax in axes.flat:
        if plotPerturbations:
            ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=1, zorder=1)
    return fig

def CombinedPlot_PropertyHistogram_ZSubsetting_HeightAverage_Perturbations(
        jointHistogramsDictionary,
        updraftType="c", t1_min=0, t2_min=20, parcelTypes=["Any"],
        labelsDictionary=labelsDictionaryPerturbation):

    [time_bins, timesteps_per_hour, z_bins] = GetCoordinateBins()
    timesteps_per_min = timesteps_per_hour/60
    property_bins_Perturbation = GetPropertyBins_Perturbations(numBins=GetNumBins_Joint())
    varNames = GetVarNames_Plotting(); varNames.remove('QV')
    parcelDepth = GetParcelDepth()

    nrows = len(parcelTypes)
    ncols = len(varNames)
    fig = plt.figure(figsize=(18, 4 * nrows))
    gs = gridspec.GridSpec(nrows, ncols + 1, width_ratios=[1]*ncols + [0.05], wspace=0.05, hspace=0.15)

    cmap = plt.get_cmap("turbo").copy()
    cmap.set_under("white")

    levels = np.arange(0, 6+0.25, 0.25) if updraftType == "c" else np.arange(0, 1+0.05, 0.05)

    im = None
    axes = []

    for i, parcelType in enumerate(parcelTypes):
        row_axes = []
        for j, varName in enumerate(varNames):

            ax = fig.add_subplot(gs[i, j]) if j == 0 else fig.add_subplot(gs[i, j], sharey=row_axes[0])
            row_axes.append(ax)
            ax.set_rasterization_zorder(1)

            multiplier = multiplierDictionary.get(varName, 1)
            units = unitsDictionary.get(varName, "")

            pairKey = f"QV_prime--{varName}_prime"
            jointHistogram4D = jointHistogramsDictionary[parcelType][parcelDepth][updraftType][pairKey]
            zMean, countHist = GetZWeightedMean(jointHistogram4D, z_bins, t1_min, t2_min, time_bins, timesteps_per_min)

            x_bins = property_bins_Perturbation[f"{varName}_prime"] * multiplier
            y_bins = property_bins_Perturbation["QV_prime"] * 1e3
            x_centers = 0.5 * (x_bins[:-1] + x_bins[1:])
            y_centers = 0.5 * (y_bins[:-1] + y_bins[1:])

            im = ax.contourf(x_centers, y_centers, zMean,
                             cmap=cmap, levels=levels, extend='both', zorder=0)

            if j == 0:
                ax.set_ylabel(f"{parcelType}\n" + r"$q_v'\ (g/kg)$")
            # if i == 0:
            #     ax.set_title(f"{labelsDictionary[varName]}")
            if i == nrows - 1:
                ax.set_xlabel(fr"{labelsDictionary[varName]} ({units})")

            ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=1, zorder=1)
            ax.axhline(0, color='black', linestyle='--', alpha=0.6, linewidth=1, zorder=1)

        for ax in row_axes[1:]:
            ax.tick_params(labelleft=False)
        axes.append(row_axes)

    cax = fig.add_subplot(gs[:, ncols])
    if im is not None:
        fig.colorbar(im, cax=cax).set_label(r"$\overline{Z}\ (km)$")

    updraftType = "General" if updraftType == 'g' else "Cloudy"
    plt.suptitle(
        f"History of Entrained Parcels ({updraftType} Updrafts) ({t1_min}-{t2_min}mins)",
        fontsize=16, y=0.95
    )
    return fig

def GetZWeightedMean(jointHistogram4D, z_bins, t1_min, t2_min, time_bins, timesteps_per_min):
    """
    Collapses a (time, z, property_1, property_2) joint histogram over a chosen
    time range into a 2D (property_1, property_2) array of mean z (grid-index),
    weighted by counts. Also returns the summed count array for masking empty bins.
    """
    countByTime = SumOverTimeRange(jointHistogram4D, time_bins, t1_min, t2_min, timesteps_per_min) #(z, prop1, prop2) -- wait, SumOverTimeRange already sums time, keeping z
    z_centers = 0.5*(z_bins[:-1] + z_bins[1:])
    zHeights_km = ModelData.zh[z_centers.astype(int)]

    zWeightedSum = (countByTime * zHeights_km[:,None,None]).sum(axis=0) #(prop1, prop2)
    totalCount = countByTime.sum(axis=0) #(prop1, prop2)

    zMean = np.divide(zWeightedSum, totalCount,
                       out=np.full_like(zWeightedSum, np.nan),
                       where=totalCount!=0)
    return zMean, totalCount

In [ ]:
# Plotting Limit Functions

def GetPropertyLimits(property_bins_Dictionary,mode):
    """
    Same as GetPropertyLimits but without the 'g' mode zoom overrides -
    those assume perturbation values centered near zero, which doesn't
    apply to raw variable values. Limits are just the full bin range.
    """
    property_limits_Dictionary = {}
 
    for varName, bins in property_bins_Dictionary.items():
        multiplier = multiplierDictionary.get(varName, 1)
        property_limits_Dictionary[varName] = (
            np.nanmin(multiplier * bins),
            np.nanmax(multiplier * bins)
        )

    if mode == "g":
        property_limits_Dictionary.update({
            "W": (-0.5, 0.5),            # m/s
        })
    # elif mode == "c":
    #     property_limits_Dictionary.update({
    #         "W": (-0.5, 0.5),            # m/s
    #     })
 
    return property_limits_Dictionary
    
def GetPropertyLimits_Perturbations(property_bins_Dictionary, mode):
    property_limits_Dictionary = {}

    for varName, bins in property_bins_Dictionary.items():
        multiplier = multiplierDictionary.get(varName, 1)
        property_limits_Dictionary[varName] = (
            np.nanmin(multiplier * bins),
            np.nanmax(multiplier * bins)
        )

    if mode == "g":
        property_limits_Dictionary.update({
            "RH_vapor": (-20, 20),   # % after multiplier
            "QV": (-1, 1),           # g/kg after multiplier
            "QCQI": (-1, 1),         # g/kg after multiplier
            "W": (-5, 5),            # m/s
            "T": (-5, 5),            # K
            "THETA_v": (-2, 2),      # K
        })

    return property_limits_Dictionary

In [ ]:
# Normalizing Functions
def NormalizeHistogram_Perturbations(histogram):
    histogram_sum = histogram.sum(axis=1, keepdims=True)
    histogram_normalized = np.divide(histogram,histogram_sum, 
                          out=np.zeros_like(histogram, dtype=float),
                          where=histogram_sum != 0)
    return histogram_normalized

def NormalizeHistogram_Perturbations_ZSubsetting(histogram):
    histogram_sum = histogram.sum(axis=0, keepdims=True) #*ZSubsetting
    histogram_normalized = np.divide(histogram,histogram_sum, 
                          out=np.zeros_like(histogram, dtype=float),
                          where=histogram_sum != 0)
    return histogram_normalized

In [ ]:
# Plotting Saving Functions
def GetPlottingDirectory(plotFileName, extension='pdf'):
    def GetPlotType():
        plotType=f"Project_Algorithms/Tracked_Profiles/EntrainmentTrackback"
        return plotType
    def GetFolderName():
        folderName=f""
        return folderName
        
    plotType = GetPlotType()
    folderName = GetFolderName()

    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    simStr = f"{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz"
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, f"Simulation_{simStr}", folderName)
    os.makedirs(specificPlottingDirectory, exist_ok=True)
    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName+f'.pdf')

    print(f'Plotting Location: {f'Saving to: {plottingFileName}'}')
    return plottingFileName

def SaveFigureToPDF(pdf,fig,plottingFileName,dpi=100):
    pdf.savefig(fig, dpi=dpi)
    plt.close(fig)

In [ ]:
###################
#PLOTTING

In [ ]:
#Plot 1: CombinedPlot_PropertyHistogram, raw values, summed over z=1-4km
if not running:
    fig = CombinedPlot_PropertyHistogram(
        histogramsDictionary_combined, updraftType="c",
        z1_km=1, z2_km=4,
        # parcelTypes=["Any"],
        parcelTypes=["CL","nonCL"],
        plotPerturbations=False
    )

In [ ]:
#Plot 2: CombinedPlot_PropertyHistogram, raw values, summed over z=1-4km
if not running:
    fig = CombinedPlot_PropertyHistogram(
        histogramsDictionary_combined, updraftType="c",
        z1_km=1, z2_km=4,
        # parcelTypes=["Any"],
        parcelTypes=["CL","nonCL"],
        plotPerturbations=True
    )

In [ ]:
#Plot 3: CombinedPlot_PropertyHistogram_ZSubsetting, summed over t=10-20 minutes back
if not running:
    fig = CombinedPlot_PropertyHistogram_ZSubsetting(
        histogramsDictionary_combined, updraftType="c",
        t1_min=10, t2_min=20,
        # parcelTypes=["Any"],
        parcelTypes=["CL","nonCL"],
        plotPerturbations=True
    )

In [ ]:
#Plot 4: CombinedPlot_PropertyHistogram_ZSubsetting_HeightAverage_Perturbations,
if not running:
    fig = CombinedPlot_PropertyHistogram_ZSubsetting_HeightAverage_Perturbations(
        jointHistogramsDictionary_combined,
        updraftType="c", t1_min=10, t2_min=20,
        # parcelTypes=["Any"],
        parcelTypes=["CL","nonCL"]
    )

In [ ]:
# #Plotting
# if not running:
#     [time_bins, _, zBins_km_g, zBins_km_c] = GetCoordinateBins()
#     for parcelTypes in [["CL","nonCL"],["SBF","ColdPool"]]:
#         plottingFileName = GetPlottingDirectory(
#             plotFileName=f'PropertyHistograms_{parcelTypes[0]}vs{parcelTypes[1]}'
#         )
#         with PdfPages(plottingFileName) as pdf:
#             for mode, zBins_km in (("g", zBins_km_g), ("c", zBins_km_c)):
#                 for z1, z2 in tqdm(zBins_km):
#                     zKey = f"{z1}_{z2}km_hist2d"
#                     fig = CombinedPlot_PropertyHistogram(histogramsDictionary_combined,
#                                                          mode, zKey, time_bins, parcelTypes, plotPerturbations=False)
#                     SaveFigureToPDF(pdf, fig, plottingFileName, dpi=200)

In [ ]:
###################
#TESTING

In [ ]:
# #testing plotting

# def TestPlotHistogram_z(varName='T'):
    
#     timesteps_per_min = timesteps_per_hour/60  #assuming timesteps_per_hour already defined from GetCoordinateBins
#     t1, t2 = 10, 20  #minutes back, range (excluding t1 itself)
    
#     property_bins=GetPropertyBins() 
#     property_bins_Perturbation=GetPropertyBins_Perturbations() 
#     qv_bins = property_bins[varName]
#     qv_centers = 0.5*(qv_bins[:-1] + qv_bins[1:])
#     qv_prime_bins = property_bins_Perturbation[f'{varName}_prime']
#     qv_prime_centers = 0.5*(qv_prime_bins[:-1] + qv_prime_bins[1:])
#     [time_bins,timesteps_per_hour,z_bins]=GetCoordinateBins()
#     z_centers = 0.5*(z_bins[:-1] + z_bins[1:])
    
#     timeIdx = np.arange(-int(timesteps_per_min*t1)-2, -int(timesteps_per_min*t2)-2, -1)
    
#     a_raw = histogramsDictionary['Any']['ALL']['c'][varName][timeIdx].sum(axis=0)
#     a_prime = histogramsDictionary['Any']['ALL']['c'][f'{varName}_prime'][timeIdx].sum(axis=0)
    
#     fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
#     axes[0].contourf(qv_centers, z_centers, a_raw, cmap='turbo')
#     axes[0].set_xlabel(varName)
#     axes[0].set_ylabel("Z level")
#     axes[0].set_title(f"Raw {varName} (0-10 min back)")
#     axes[1].contourf(qv_prime_centers, z_centers, a_prime, cmap='turbo')
#     axes[1].set_xlabel(f"{varName}' (perturbation)")
#     axes[1].set_title(f"{varName} Perturbation (0-10 min back)")
#     plt.tight_layout()

# def TestPlotHistogram_t(varName='T'):
    
#     z1, z2 = 1, 4   #z levels, inclusive
    
#     property_bins=GetPropertyBins() 
#     property_bins_Perturbation=GetPropertyBins_Perturbations() 
#     qv_bins = property_bins[varName]
#     qv_centers = 0.5*(qv_bins[:-1] + qv_bins[1:])
#     qv_prime_bins = property_bins_Perturbation[f'{varName}_prime']
#     qv_prime_centers = 0.5*(qv_prime_bins[:-1] + qv_prime_bins[1:])
#     [time_bins,timesteps_per_hour,z_bins]=GetCoordinateBins()
#     time_centers = 0.5*(time_bins[:-1] + time_bins[1:])
    
#     a_raw = histogramsDictionary['Any']['ALL']['c'][varName][:, z1:z2+1, :].sum(axis=1)
#     a_prime = histogramsDictionary['Any']['ALL']['c'][f'{varName}_prime'][:, z1:z2+1, :].sum(axis=1)
    
#     fig, axes = plt.subplots(1, 2, figsize=(10,4))
#     axes[0].contourf(time_centers, qv_centers, a_raw.T, cmap='turbo')
#     axes[0].set_xlabel("Time (relative minutes)")
#     axes[0].set_ylabel(varName)
#     axes[0].set_title(f"Raw {varName} (z 1-4)")
#     axes[1].contourf(time_centers, qv_prime_centers, a_prime.T, cmap='turbo')
#     axes[1].set_xlabel("Time (relative minutes)")
#     axes[1].set_ylabel(f"{varName}' (perturbation)")
#     axes[1].set_title(f"{varName} Perturbation (z 1-4)")
#     plt.tight_layout()

# #joint plot: QV' vs T', colored by average Z4
# def TestPlotHistogram_Joint(varName='T'):
#     varName='T'
    
#     [time_bins,timesteps_per_hour,z_bins]=GetCoordinateBins()
#     time_centers = 0.5*(time_bins[:-1] + time_bins[1:])
#     z_centers = 0.5*(z_bins[:-1] + z_bins[1:])
#     zHeights_km = ModelData.zh[:len(z_centers)]
    
#     property_bins_Perturbation=GetPropertyBins_Perturbations()
#     qv_prime_bins = property_bins_Perturbation['QV_prime']
#     qv_prime_centers = 0.5*(qv_prime_bins[:-1] + qv_prime_bins[1:])
#     t_prime_bins = property_bins_Perturbation[f'{varName}_prime']
#     t_prime_centers = 0.5*(t_prime_bins[:-1] + t_prime_bins[1:])
#     pairKey = f"QV_prime--{varName}_prime"
#     jointCounts = jointHistogramsDictionary['Any']['ALL']['c'][pairKey] #shape (time, z, QV'_bins, T'_bins)
#     countByZ = jointCounts.sum(axis=0) #shape (z, QV'_bins, T'_bins)
#     zWeightedSum = (countByZ * zHeights_km[:,None,None]).sum(axis=0) #shape (QV'_bins, T'_bins)
#     totalCount = countByZ.sum(axis=0) #shape (QV'_bins, T'_bins)
#     zMean = np.divide(zWeightedSum, totalCount,
#                        out=np.full_like(zWeightedSum, np.nan),
#                        where=totalCount!=0)
    
#     levels = np.arange(0, 6+0.25, 0.25)
#     cmap = plt.get_cmap('turbo').copy()
#     cmap.set_under('white') #anything below the lowest level (incl. NaN-adjacent zero regions) renders white
    
#     plt.contourf(t_prime_centers, qv_prime_centers, zMean, levels=levels, cmap=cmap, extend='both')
#     plt.colorbar(label="Mean Z height (km)")
#     plt.xlabel(f"{varName}' (perturbation)")
#     plt.ylabel("QV' (perturbation)")
#     plt.title(f"QV' vs {varName}' joint distribution, colored by mean Z")